In [ ]:
# Prerequisite - Role A - Data loading.
# Simulation

# Requirements:
# pip install psycopg2-binary
# pip install sqlalchemy
import sqlalchemy as sa

from dotenv import load_dotenv
import os
import pandas as pd

# Load configuration from .env file
load_dotenv(override=True)

# Connect to Supabase directly using SQLAlchemy.
supabase_direct = sa.create_engine(os.getenv("SUPABASE_CONNECTION_STRING"))

# Loading data using SQL query over direct connection.
def load_data_using_sql(sql):
    query = sa.text(sql)
    return pd.read_sql(query, supabase_direct)
    
# Load all data from sales and customers tables.
df_sales = load_data_using_sql("SELECT * FROM sales;")
df_customers = load_data_using_sql("SELECT * FROM customers;")

# Merge sales and customers.
df = pd.merge(df_sales, df_customers, on="customer_id", how="left")

# Overview of result (will be printed out in Jupyter):
df_sales.shape, df_customers.shape, df.shape

In [ ]:
# Prerequisite - Role B - Data cleaning.
# Nothing particular needed to be done here for RFM analysis to be possible.
last_sales_before_2025_anomaly = load_data_using_sql(
    "SELECT * FROM sales WHERE sale_date >= '2025-02-01' AND sale_date < '2025-12-01' ORDER BY sale_date DESC LIMIT 5"
)
last_sales_before_2025_anomaly.head()

In [ ]:
# My role: Role C - RFM analysis.

# reference_date corresponds to "today" in the teamwork manual.
reference_date = pd.to_datetime("2025-02-28")

df_for_rfm = df[df["sale_date"] <= reference_date]

rfm = df_for_rfm.groupby("customer_id").agg(
    last_purchase=("sale_date", "max"),
    total_purchases=("id", "size"),
    total_spending=("total_price", "sum")
).reset_index()

rfm["days_since_last_purchase"] = (reference_date - rfm["last_purchase"]).dt.days

q = 5 # number of quantiles
ratings = list(range(1, q + 1)) # for F and M scores
reversed_ratings = list(range(q, 0, -1)) # for R score - the smaller the days_since_last_purchase value, the higher the rating
print("Possible R ratings:", reversed_ratings)
print("Possible F and M ratings:", ratings)

# R score (recency).
rfm["R_score"] = pd.qcut(rfm["days_since_last_purchase"].rank(method="first"), q, labels=reversed_ratings).astype(int)

# F score (frequency).
# Encountered an error: ValueError: Bin edges must be unique: Index([1.0, 2.0, 2.0, 3.0, 5.0, 77.0], dtype='float64', name='total_purchases').
# AI suggested to use .rank(method="first") to fix the problem.
rfm["F_score"] = pd.qcut(rfm["total_purchases"].rank(method="first"), q, labels=ratings).astype(int)

# M score (monetary).
rfm["M_score"] = pd.qcut(rfm["total_spending"].rank(method="first"), q, labels=ratings).astype(int)

# RFM score as sum of individual scores.
rfm["RFM_score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

# Segmentation.
# 13-15 = VIP Champions, 10-12 = Loyal, 7-9 = Potential, 4-6 = At Risk, 3 = Lost.
# Return numeric values to allow meaningful sorting.
def to_segment_value(score):
    if score >= 13:
        return 4
    elif score >= 10:
        return 3
    elif score >= 7:
        return 2
    elif score >= 4:
        return 1
    else:
        return 0

segment_labels = ("Lost", "At Risk", "Potential", "Loyal", "VIP Champions")

rfm["segment_value"] = rfm["RFM_score"].apply(to_segment_value)
rfm["segment"] = rfm["segment_value"].apply(lambda x: segment_labels[x])

segments = rfm.groupby(["segment", "segment_value"]).agg(
    total_customers=("segment_value", "size"),
    total_purchases=("total_purchases", "sum"),
    total_spending=("total_spending", "sum"),
    average_spending=("total_spending", "mean")
).sort_values("segment_value", ascending=False).reset_index()

segments["total_spending_share"] = segments["total_spending"] / segments["total_spending"].sum()

segments # This will pretty-print the result in Jupyter notebook.

In [ ]:
# Role D - Visualization.
# Just a simplified simulation.

import plotly.express as px

fig1 = px.bar(
    segments,
    title="Klientide arv erinevates segmentides",
    x="segment",
    y="total_customers",
    labels={"segment": "Segment", "total_customers": "Klientide arv"}
)

fig1.update_layout(
    plot_bgcolor="white"
)

fig1.show()

fig2 = px.scatter(
    rfm,
    title="Klientide kogukulutused võrdluses viimasest ostust möödunud päevade arvuga",
    x="days_since_last_purchase",
    y="total_spending",
    color="segment",
    size="total_purchases",
    hover_data=["customer_id"],
    labels={
        "segment": "Segment",
        "days_since_last_purchase": "Päevade arv viimasest ostust",
        "total_spending": "Kogukulutus",
        "total_purchases": "Tellimuste arv",
        "customer_id": "Kliendi ID"
    }
)

fig2.update_layout(
    plot_bgcolor="white"
)

fig2.show()

df_for_fig3 = rfm[rfm["segment"] == "VIP Champions"].nlargest(10, "total_spending").sort_values("total_spending", ascending=False)

fig3 = px.bar(
    df_for_fig3,
    title="TOP 10 klienti VIP Champions segmendis",
    x="customer_id",
    y="total_spending",
    labels={"customer_id": "Kliendi ID", "total_spending": "Kogukulutus"}
)

fig3.update_xaxes(type='category')

fig3.update_layout(
    plot_bgcolor="white"
)

fig3.show()